# Organizer submission budget: installed-source CPU probe

Use Accelerator None, Internet off, the official wheelhouse and compatible
Linux x86_64 CPython 3.12. This notebook copies the successful notebook 14 CPU
setup definitions, checks the unchanged organizer versions and verifier hash,
then scans only named installed package Python sources for budget definitions.
No model, task snapshot, problem statement, reference/test patch, evaluator run
or grader run is loaded. Discovered modules are not imported. An optional loader
call requires an already-loaded matching plain function with an observed single
config-root argument and conservative config-only body; otherwise retain source.

Stage1 script/inference.py may live outside the installed packages. Missing loader
source is a recorded scope limit, not schema validation. No full runtime budget,
environment-variable precedence or main submission selection is established.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, base64, zipfile, io, time, inspect
IS_KAGGLE = Path('/kaggle/working').is_dir()
RUN_DECLARED_EXPERIMENT = True
MODE = 'cpu_controls'
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
if not IS_KAGGLE:
    print('Local definitions only: no installation, task read, inference or grader call')


In [ ]:
"""Export only artifacts produced inside a run, including per-trial checkpoints."""
import base64
import hashlib
import io
import json
import os
from pathlib import Path
import tempfile
import zipfile


def export_run(run_root: Path, *, emit: bool = True) -> dict:
    run_root = run_root.resolve(strict=True)
    artifacts, manifest = [], []
    for artifact in sorted(run_root.rglob('*')):
        resolved = artifact.resolve()
        if artifact.is_symlink() or not resolved.is_relative_to(run_root):
            raise ValueError('Run artifact is a link or escaped its directory')
        name = artifact.relative_to(run_root).as_posix()
        if name == 'evidence_bundle.zip' or name.startswith('evidence_checkpoints/'):
            continue
        if not artifact.is_file():
            continue
        if name == 'artifact_manifest.json':
            raise ValueError('artifact_manifest.json is reserved for the exported manifest')
        # Read once. A live log may grow, but its archived bytes and hash must agree.
        payload = artifact.read_bytes()
        artifacts.append((name, payload))
        manifest.append({'path': name, 'bytes': len(payload),
                         'sha256': hashlib.sha256(payload).hexdigest()})
    if sum(item['bytes'] for item in manifest) > 32 * 1024 * 1024:
        raise ValueError('Use Kaggle output download for this large evidence bundle')
    bundle = io.BytesIO()
    with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
        for name, payload in artifacts:
            archive.writestr(name, payload)
        archive.writestr('artifact_manifest.json', json.dumps(manifest, indent=2) + '\n')
    bundle_bytes = bundle.getvalue()
    digest = hashlib.sha256(bundle_bytes).hexdigest()
    checkpoints = run_root / 'evidence_checkpoints'
    checkpoints.mkdir(exist_ok=True)
    checkpoint = checkpoints / (digest + '.zip')
    if checkpoint.exists():
        if checkpoint.read_bytes() != bundle_bytes:
            raise ValueError('Existing checkpoint differs from its content hash')
    else:
        with checkpoint.open('xb') as stream:
            stream.write(bundle_bytes)
            stream.flush()
            os.fsync(stream.fileno())
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(dir=run_root, prefix='.evidence_bundle.',
                                         suffix='.tmp', delete=False) as stream:
            temporary = Path(stream.name)
            stream.write(bundle_bytes)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, run_root / 'evidence_bundle.zip')
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()
    report = {'sha256': digest, 'bytes': len(bundle_bytes), 'file_count': len(manifest),
              'checkpoint_path': checkpoint.relative_to(run_root).as_posix(),
              'snapshot_is_atomic_across_all_files': False}
    if emit:
        print('NEURAL_SWE_BUNDLE_SHA256=' + report['sha256'], flush=True)
        print('NEURAL_SWE_BUNDLE_BASE64=' + base64.b64encode(bundle_bytes).decode('ascii'), flush=True)
    return report


In [ ]:
PLAN = json.loads('{"scope": "installed_organizer_budget_source_CPU_probe_no_tasks_or_model", "created_utc": "2026-10-04T19:01:28.979899+00:00", "expected_core_packages": {"swegemma": "0.2.7", "adk-submission": "0.2.12", "adk-eval-core": "0.1.0"}, "expected_verify_task_sha256": "98600fd6066a4dbd472490cb194b839a3a633afea9b99256cc081ea77beac0d3", "definition_scope": "named_distributions_package_python_only", "limitations": "Stage1 inference.py may be external to these packages. Absence does not certify or disprove its sidecar loader."}')
PLAN_BYTES_BASE64 = 'ewogICJzY29wZSI6ICJpbnN0YWxsZWRfb3JnYW5pemVyX2J1ZGdldF9zb3VyY2VfQ1BVX3Byb2JlX25vX3Rhc2tzX29yX21vZGVsIiwKICAiY3JlYXRlZF91dGMiOiAiMjAyNi0xMC0wNFQxOTowMToyOC45Nzk4OTkrMDA6MDAiLAogICJleHBlY3RlZF9jb3JlX3BhY2thZ2VzIjogewogICAgInN3ZWdlbW1hIjogIjAuMi43IiwKICAgICJhZGstc3VibWlzc2lvbiI6ICIwLjIuMTIiLAogICAgImFkay1ldmFsLWNvcmUiOiAiMC4xLjAiCiAgfSwKICAiZXhwZWN0ZWRfdmVyaWZ5X3Rhc2tfc2hhMjU2IjogIjk4NjAwZmQ2MDY2YTRkYmQ0NzI0OTBjYjE5NGI4MzlhM2E2MzNhZmVhOWI5OTI1NmNjMDgxZWE3N2JlYWMwZDMiLAogICJkZWZpbml0aW9uX3Njb3BlIjogIm5hbWVkX2Rpc3RyaWJ1dGlvbnNfcGFja2FnZV9weXRob25fb25seSIsCiAgImxpbWl0YXRpb25zIjogIlN0YWdlMSBpbmZlcmVuY2UucHkgbWF5IGJlIGV4dGVybmFsIHRvIHRoZXNlIHBhY2thZ2VzLiBBYnNlbmNlIGRvZXMgbm90IGNlcnRpZnkgb3IgZGlzcHJvdmUgaXRzIHNpZGVjYXIgbG9hZGVyLiIKfQo='
SOURCE_BINDING = json.loads('{"parent_environment_notebook": "kaggle/14_public_fixture_controls.ipynb", "parent_notebook_sha256": "c18a70ef8c0af3d370f0b13d5a69c6c25fe3b367bb96dad61ed049c77b4d286d", "probe_module_sha256": "a3f9f7eb1d1713771c94693210baa4576018e1ca22b4bb61c51a08aa946a26e5", "builder_sha256": "4e61b3a9415c9ee1969fc377f0a759bbd60e66f703ebc5e70e03edfe53143200", "plan_sha256": "bd95def0c97acd8878c567107f5b382bf968d5909927cf70fd595efc48984abe", "copied_parent_cells": [2, 4, 5, 6]}')
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    RUN = Path('/kaggle/working') / ('budget_source_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f'))
    RUN.mkdir()
    frozen = base64.b64decode(PLAN_BYTES_BASE64, validate=True)
    if hashlib.sha256(frozen).hexdigest() != SOURCE_BINDING['plan_sha256'] or json.loads(frozen) != PLAN:
        raise ValueError('Frozen plan/source binding differs')
    (RUN / 'plan.json').write_bytes(frozen)
    (RUN / 'source_binding.json').write_text(json.dumps(SOURCE_BINDING, indent=2) + '\n', encoding='utf-8')
    export_run(RUN)


In [ ]:
"""Fail-fast, recorded CPU setup for the unchanged organizer source audit.

Definitions do not install packages. Runtime checks never filter incompatible
wheels into a superficially successful setup or establish scoring equivalence.
"""
from datetime import datetime, timezone
import glob
import hashlib
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
import sysconfig
import traceback


REQUIRED_CPU_RUNTIME = {
    "implementation": "CPython", "python_major_minor": [3, 12],
    "system": "Linux", "machine": "x86_64",
}


def environment_file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        while chunk := stream.read(1048576):
            digest.update(chunk)
    return digest.hexdigest()


def environment_package_versions(names):
    values = {}
    for name in sorted(names):
        try:
            values[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            values[name] = None
    return values


def cpu_runtime_snapshot(package_names):
    return {
        "implementation": platform.python_implementation(),
        "python_major_minor": list(sys.version_info[:2]),
        "python_version": sys.version, "executable": sys.executable,
        "executable_realpath": str(Path(sys.executable).resolve()),
        "system": platform.system(), "machine": platform.machine(),
        "platform": platform.platform(), "libc": list(platform.libc_ver()),
        "soabi": sysconfig.get_config_var("SOABI"),
        "multiarch": sysconfig.get_config_var("MULTIARCH"),
        "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
        "package_versions": environment_package_versions(package_names),
    }


def cpu_runtime_errors(runtime):
    return [key + " must be " + repr(expected) + "; observed " + repr(runtime.get(key))
            for key, expected in REQUIRED_CPU_RUNTIME.items()
            if runtime.get(key) != expected]


def normalized_organizer_wheel_name(name):
    # Retain the organizer workaround already used in frozen notebook 10.
    return name.replace("cu128", "+cu128") if "cu128" in name and "+" not in name else name


def classify_organizer_wheels(paths, supported_tags=None):
    from packaging.tags import sys_tags
    from packaging.utils import parse_wheel_filename
    supported = set(sys_tags() if supported_tags is None else supported_tags)
    rows = []
    for path in sorted(map(Path, paths), key=lambda item: item.name):
        if not path.is_file():
            raise ValueError("Missing organizer wheel: " + str(path))
        row = {"source_path": str(path), "original_name": path.name,
               "install_name": normalized_organizer_wheel_name(path.name),
               "bytes": path.stat().st_size, "sha256": environment_file_sha256(path)}
        if "cutlass" in path.name.lower():
            row.update(selected=False, compatible=None,
                       exclusion_reason="existing_organizer_cutlass_workaround")
        else:
            try:
                distribution, version, build, tags = parse_wheel_filename(row["install_name"])
                row.update(selected=True, compatible=bool(tags & supported),
                           distribution=distribution, version=str(version),
                           wheel_tags=sorted(map(str, tags)),
                           matching_runtime_tags=sorted(map(str, tags & supported)))
            except ValueError as error:
                row.update(selected=True, compatible=False, wheel_parse_error=str(error))
        rows.append(row)
    if len({row["install_name"] for row in rows}) != len(rows):
        raise ValueError("Ambiguous organizer wheel aliases")
    return rows


def remove_recorded_cutlass_hooks():
    removed = []
    for pattern in ("/usr/local/lib/python*/dist-packages/*cutlass*.pth",
                    "/usr/local/lib/python*/site-packages/*cutlass*.pth"):
        for name in sorted(glob.glob(pattern)):
            path = Path(name)
            row = {"path": str(path), "sha256": environment_file_sha256(path)}
            path.unlink()
            removed.append(row)
    return removed


def source_audit_import_check():
    import torch
    from swegemma.config import EvalConfig
    from swegemma.evaluate import Evaluator
    from swegemma.models import load_tasks
    count = torch.cuda.device_count()
    if count != 0:
        raise ValueError("Select Accelerator None: source audit requires zero CUDA devices")
    return {"torch_version": torch.__version__, "cuda_device_count": count,
            "required_organizer_imports_succeeded": True}


def prepare_source_audit_environment(wheelhouse_dir, run_root, expected_packages,
                                     checkpoint_exporter, source_binding):
    """Install the whole selected organizer wheelhouse only after compatibility.

    Always retain runtime identity and setup failures before allowing the exact
    source inventory. UI labels are recorded separately from verified runtime.
    """
    run_root = Path(run_root)
    wheelhouse_dir = Path(wheelhouse_dir)
    run_root.mkdir(parents=True, exist_ok=False)
    package_names = set(expected_packages) | {"torch", "vllm", "pip", "packaging"}
    report = {
        "scope": "CPU_source_audit_environment_setup_not_model_or_grading",
        "created_utc": datetime.now(timezone.utc).isoformat(), "status": "started",
        "required_runtime": REQUIRED_CPU_RUNTIME, "source_binding": source_binding,
        "before": cpu_runtime_snapshot(package_names), "after": None,
        "wheelhouse_path": str(wheelhouse_dir), "wheel_manifest": [],
        "selected_wheels_omitted_for_incompatibility": [],
        "cutlass_hooks_removed": [], "pip_command": None, "pip_return_code": None,
        "model_inference_executed": False, "source_inventory_executed": False,
        "tests_executed": False, "private_scoring_equivalence_established": False,
        "runtime_image_id_verified": False,
        "requested_UI_profile": "original 2026-07-01 image, Accelerator None, Internet off",
    }

    def save():
        (run_root / "environment_receipt.json").write_text(
            json.dumps(report, indent=2) + "\n", encoding="utf-8")
        print("MOBILE_SOURCE_AUDIT_ENVIRONMENT=" + json.dumps({
            "status": report["status"], "receipt": str(run_root / "environment_receipt.json"),
            "python_version": report["before"]["python_version"],
            "runtime_errors": report.get("runtime_errors", []),
            "incompatible_wheels": report.get("incompatible_wheels", []),
            "pip_return_code": report["pip_return_code"],
        }), flush=True)
        checkpoint_exporter(run_root)

    save()
    try:
        if not wheelhouse_dir.is_dir():
            raise RuntimeError("Attach the official organizer wheelhouse before setup")
        report["wheel_manifest"] = classify_organizer_wheels(wheelhouse_dir.glob("*.whl"))
        selected = [row for row in report["wheel_manifest"] if row["selected"]]
        report["runtime_errors"] = cpu_runtime_errors(report["before"])
        report["incompatible_wheels"] = [row["original_name"] for row in selected if not row["compatible"]]
        if not selected or report["runtime_errors"] or report["incompatible_wheels"]:
            report["status"] = "preflight_rejected_before_install"
            raise RuntimeError("CPU source-audit preflight rejected this environment; "
                               "use a verified Linux x86_64 CPython 3.12 worker. "
                               "Incompatible organizer wheels were retained and no install ran.")
        report["status"] = "preflight_passed"
        save()
        links = run_root / "organizer_wheel_links"
        links.mkdir()
        for row in selected:
            (links / row["install_name"]).symlink_to(Path(row["source_path"]).resolve())
        report["cutlass_hooks_removed"] = remove_recorded_cutlass_hooks()
        command = [sys.executable, "-m", "pip", "install", "-q", "--no-index",
                   "--no-deps", "--force-reinstall"] + [str(links / row["install_name"]) for row in selected]
        report.update(status="installing", pip_command=command)
        save()
        result = subprocess.run(command, capture_output=True, text=True)
        (run_root / "organizer_install_stdout.txt").write_text(result.stdout or "", encoding="utf-8")
        (run_root / "organizer_install_stderr.txt").write_text(result.stderr or "", encoding="utf-8")
        report["pip_return_code"] = result.returncode
        importlib.invalidate_caches()
        report["after"] = cpu_runtime_snapshot(package_names)
        if result.returncode != 0:
            report["status"] = "organizer_install_failed"
            raise RuntimeError("Organizer wheel installation failed; inspect retained install logs")
        observed = {name: report["after"]["package_versions"][name] for name in expected_packages}
        if observed != expected_packages:
            report["status"] = "organizer_core_identity_rejected"
            raise ValueError("Installed organizer core packages differ from the frozen source-audit plan")
        report["status"] = "checking_CPU_organizer_imports"
        save()
        report.update(source_audit_import_check())
        report["status"] = "ready_for_unchanged_source_audit"
        save()
        return report
    except Exception as error:
        if report["status"] in {"started", "preflight_passed", "installing", "checking_CPU_organizer_imports"}:
            report["status"] = "setup_exception_before_source_audit"
        report.update(error_type=type(error).__name__, error=str(error))
        (run_root / "setup_exception.txt").write_text(traceback.format_exc(), encoding="utf-8")
        save()
        raise


In [ ]:
def resolve_organizer_audit_api(evaluator_module):
    verifier = getattr(evaluator_module, "verify_task", None)
    if not callable(verifier):
        raise ValueError("Installed Evaluator module has no callable official verify_task")
    implementation = inspect.getmodule(verifier)
    namespace = getattr(verifier, "__globals__", {})
    apis = {"verify_task": verifier}
    for name in ("resolve_task_snapshot_paths", "extract_snapshot", "sandbox_start", "sandbox_exec", "sandbox_stop"):
        # verify_task's own globals are the callables used by its retained source.
        # Snapshot resolution can be a public reexport from Evaluator's module.
        function = namespace.get(name)
        if function is None and implementation is not None:
            function = getattr(implementation, name, None)
        if function is None and name == "resolve_task_snapshot_paths":
            function = getattr(evaluator_module, name, None)
        if not callable(function):
            raise ValueError("Official verifier implementation lacks callable " + name)
        apis[name] = function
    placeholder = object()
    calls = {
        "resolve_task_snapshot_paths": ((placeholder, "instance_id", "repo"), {}),
        "extract_snapshot": ((placeholder, placeholder, placeholder),
                             {"patch_path": placeholder, "base_snapshot_path": placeholder}),
        "sandbox_start": ((placeholder,), {}),
        "sandbox_exec": ((placeholder, placeholder, "command"), {}),
        "sandbox_stop": ((placeholder, placeholder), {}),
    }
    for name, (args, kwargs) in calls.items():
        inspect.signature(apis[name]).bind(*args, **kwargs)
    for name in ("sandbox_start", "sandbox_exec", "sandbox_stop"):
        if not inspect.iscoroutinefunction(apis[name]):
            raise ValueError("Official sandbox callable is not asynchronous: " + name)
    return apis


In [ ]:
"""Recorded CPU/GPU preflight for separately frozen future public comparisons."""
from datetime import datetime, timezone
import hashlib
import importlib
import inspect
import json
from pathlib import Path
import subprocess
import sys
import tempfile
import traceback



def declared_backend_check(mode, torch):
    count = torch.cuda.device_count()
    names = [torch.cuda.get_device_name(index) for index in range(count)]
    if mode == 'cpu_controls':
        if count != 0:
            raise ValueError('Select Accelerator None for fixture controls')
    elif mode == 'gpu_comparison':
        if count != 4 or any('L4' not in name for name in names):
            raise ValueError('The declared comparison requires exactly four NVIDIA L4 GPUs')
    else:
        raise ValueError('Unknown declared experiment mode')
    return {'cuda_device_count': count, 'gpu_names': names, 'torch_version': torch.__version__}


def prepare_declared_environment(wheelhouse_dir, run_root, plan, checkpoint_exporter, mode):
    run_root, wheelhouse_dir = Path(run_root), Path(wheelhouse_dir)
    receipt = run_root / 'environment_receipt.json'
    if receipt.exists():
        raise FileExistsError('Preserve the existing environment attempt')
    expected = plan['expected_core_packages']
    names = set(expected) | {'torch', 'vllm', 'pip', 'packaging'}
    report = {'created_utc': datetime.now(timezone.utc).isoformat(),
              'scope': 'declared_public_comparison_environment_not_model_or_control_results',
              'mode': mode, 'status': 'started', 'before': cpu_runtime_snapshot(names),
              'after': None, 'wheel_manifest': [], 'incompatible_wheels': [],
              'selected_wheels_omitted_for_incompatibility': [], 'pip_return_code': None,
              'plan_sha256': environment_file_sha256(run_root / 'plan.json'),
              'model_inference_executed': False, 'fixture_controls_executed': False,
              'runtime_image_id_verified': False}
    def save():
        receipt.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
        print('DECLARED_ENVIRONMENT=' + json.dumps({key: report.get(key) for key in
            ('mode', 'status', 'runtime_errors', 'incompatible_wheels', 'pip_return_code')}), flush=True)
        checkpoint_exporter(run_root)
    save()
    try:
        if not wheelhouse_dir.is_dir():
            raise ValueError('Attach the complete official organizer wheelhouse')
        report['runtime_errors'] = cpu_runtime_errors(report['before'])
        report['wheel_manifest'] = classify_organizer_wheels(wheelhouse_dir.glob('*.whl'))
        selected = [row for row in report['wheel_manifest'] if row['selected']]
        report['incompatible_wheels'] = [row['original_name'] for row in selected if not row['compatible']]
        if not selected or report['runtime_errors'] or report['incompatible_wheels']:
            report['status'] = 'preflight_rejected_before_install'
            raise RuntimeError('Use a compatible Linux x86_64 CPython 3.12 organizer worker; no incompatible wheels were omitted or installed')
        report['cutlass_hooks_removed'] = remove_recorded_cutlass_hooks()
        # Link staging stays outside the exported run; its wheel identities remain
        # in the receipt. It cannot collide with strict artifact link rejection.
        with tempfile.TemporaryDirectory(prefix='neural_swe_declared_wheels_') as temporary:
            links = Path(temporary)
            for row in selected:
                (links / row['install_name']).symlink_to(Path(row['source_path']).resolve())
            command = [sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
                       '--no-deps', '--force-reinstall'] + [str(links / row['install_name']) for row in selected]
            report.update(status='installing', pip_command=command)
            save()
            result = subprocess.run(command, capture_output=True, text=True, timeout=1200)
            (run_root / 'organizer_install_stdout.txt').write_text(result.stdout or '', encoding='utf-8')
            (run_root / 'organizer_install_stderr.txt').write_text(result.stderr or '', encoding='utf-8')
            report['pip_return_code'] = result.returncode
        importlib.invalidate_caches()
        report['after'] = cpu_runtime_snapshot(names)
        if result.returncode != 0:
            raise RuntimeError('Organizer installation failed; retained install logs are available')
        if {name: report['after']['package_versions'][name] for name in expected} != expected:
            raise ValueError('Installed organizer core identity differs from the frozen plan')
        import torch
        from swegemma.config import EvalConfig
        from swegemma.evaluate import Evaluator
        from swegemma.models import load_tasks
        report.update(declared_backend_check(mode, torch))
        apis = resolve_organizer_audit_api(inspect.getmodule(Evaluator))
        verifier_sha = hashlib.sha256(inspect.getsource(apis['verify_task']).encode()).hexdigest()
        report['verify_task_sha256'] = verifier_sha
        if verifier_sha != plan['expected_verify_task_sha256']:
            raise ValueError('Actual organizer verifier differs from the frozen contract')
        report['status'] = 'ready_for_' + mode
        save()
        return report
    except Exception as error:
        if report['status'] != 'preflight_rejected_before_install':
            report['status'] = 'setup_exception_before_experiment'
        report.update(error_type=type(error).__name__, error=str(error))
        (run_root / 'setup_exception.txt').write_text(traceback.format_exc(), encoding='utf-8')
        save()
        raise


In [ ]:
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    prepare_declared_environment(WHEELHOUSE_DIR, RUN, PLAN, export_run, MODE)
else:
    print('No organizer installation executed locally')


In [ ]:
"""Collect installed organizer budget-contract source without tasks or models.

Only named organizer distributions' package .py files are scanned. A loader is
called only if already loaded, its observed signature is unambiguous, and its
source has a conservative config-only call/path surface. Otherwise retain source.
"""
import ast
import builtins
from contextlib import redirect_stderr, redirect_stdout
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import inspect
import io
import json
import math
from pathlib import Path, PurePosixPath
import sys
import tempfile
import tokenize
import traceback


ORGANIZER_PACKAGES = {"swegemma": "swegemma", "adk-submission": "adk_submission", "adk-eval-core": "adk_eval_core"}
TARGET_CLASSES = {"EvalConfig", "EvaluationBudget", "HarnessLimits", "SubmissionEvalConfig", "EvaluationConfig"}
LOADER_NAME = "load_submission_eval_config"
MAX_SOURCE_BYTES = 32 * 1024 * 1024
MAX_FILE_BYTES = 2 * 1024 * 1024
POSITIVE_CONFIG = {"timeout_seconds": 300, "max_time_minutes": 20, "max_tool_calls": 50, "max_turns": 100}
POSITIVE_CONFIG_BYTES = b"evaluation:\n  timeout_seconds: 300\n  max_time_minutes: 20\n  max_tool_calls: 50\n  max_turns: 100\n"


def digest(payload):
    return hashlib.sha256(payload).hexdigest()


def module_for_file(prefix, relative):
    parts = PurePosixPath(relative).parts
    if (not parts or parts[0] != prefix or any(part in {"", ".", ".."} for part in parts)
            or PurePosixPath(relative).suffix != ".py"):
        raise ValueError("Source member is outside the declared organizer package")
    pieces = list(parts)
    if pieces[-1] == "__init__.py":
        pieces.pop()
    else:
        pieces[-1] = pieces[-1][:-3]
    return ".".join(pieces)


def definition_source(source, node):
    first = min([node.lineno] + [decorator.lineno for decorator in getattr(node, "decorator_list", [])])
    return "".join(source.splitlines(keepends=True)[first - 1:node.end_lineno])


def scan_organizer_sources(destination, expected_packages, *, distribution_reader=None):
    distribution_reader = distribution_reader or importlib.metadata.distribution
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=False)
    inventory, definitions, retained, total = [], [], {}, 0
    for name, prefix in ORGANIZER_PACKAGES.items():
        distribution = distribution_reader(name)
        if distribution.version != expected_packages[name] or distribution.files is None:
            raise ValueError("Installed organizer distribution differs or has no file inventory")
        package_root = Path(distribution.locate_file(prefix)).resolve(strict=True)
        entries = sorted(str(member) for member in distribution.files
                         if str(member).startswith(prefix + "/") and str(member).endswith(".py"))
        if not entries:
            raise ValueError("Organizer distribution has no installed package Python sources")
        for relative in entries:
            if len(inventory) >= 10000:
                raise ValueError("Organizer source inventory exceeds the bounded file count")
            module = module_for_file(prefix, relative)
            path = Path(distribution.locate_file(relative))
            cursor = path
            while cursor != package_root and package_root in cursor.parents:
                if cursor.is_symlink():
                    raise ValueError("Organizer package source is a link")
                cursor = cursor.parent
            if not path.resolve(strict=True).is_relative_to(package_root) or not path.is_file():
                raise ValueError("Organizer source escaped its package root")
            if path.stat().st_size > MAX_FILE_BYTES:
                raise ValueError("Organizer source file exceeds the bounded collection capacity")
            payload = path.read_bytes()
            total += len(payload)
            if total > MAX_SOURCE_BYTES:
                raise ValueError("Organizer source inventory exceeds the bounded byte capacity")
            encoding, _ = tokenize.detect_encoding(io.BytesIO(payload).readline)
            source = payload.decode(encoding)
            tree = ast.parse(source, filename=str(path))
            row = {"distribution": name, "version": distribution.version, "package": prefix,
                   "package_relative_path": relative, "actual_path": str(path.resolve()),
                   "module": module, "bytes": len(payload), "sha256": digest(payload)}
            inventory.append(row)
            matches = [node for node in tree.body
                       if (isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == LOADER_NAME)
                       or (isinstance(node, ast.ClassDef) and node.name in TARGET_CLASSES)]
            if not matches:
                continue
            retained_name = "organizer_sources/" + name.replace("-", "_") + "/" + relative
            retained_path = destination.parent / retained_name
            retained_path.parent.mkdir(parents=True, exist_ok=True)
            retained_path.write_bytes(payload)
            retained[retained_name] = row
            for node in matches:
                body = definition_source(source, node)
                definition_name = "organizer_definitions/" + name.replace("-", "_") + "/" + relative[:-3] + "/" + node.name + ".py"
                definition_path = destination.parent / definition_name
                definition_path.parent.mkdir(parents=True, exist_ok=True)
                definition_path.write_bytes(body.encode("utf-8"))
                definitions.append({**row, "name": node.name, "kind": type(node).__name__,
                    "first_line": min([node.lineno] + [d.lineno for d in getattr(node, "decorator_list", [])]),
                    "last_line": node.end_lineno, "decorated": bool(getattr(node, "decorator_list", [])),
                    "retained_file": retained_name, "retained_definition": definition_name,
                    "definition_sha256": digest(body.encode("utf-8")),
                    "source_signature": ast.unparse(node.args) if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) else None,
                    "return_annotation": ast.unparse(node.returns) if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.returns else None})
    result = {"scope": "only_named_organizer_distribution_package_python_sources",
              "distribution_versions": expected_packages, "file_count": len(inventory), "source_bytes": total,
              "files": inventory, "definitions": definitions, "retained_files": retained,
              "task_snapshot_or_oracle_files_scanned": False, "arbitrary_module_imports_performed": False}
    (destination / "package_python_inventory.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    return result


def config_path_role(tree, parameter):
    def parameter_path(node):
        return (isinstance(node, ast.Name) and node.id == parameter
                or isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == "Path"
                and len(node.args) == 1 and not node.keywords and isinstance(node.args[0], ast.Name)
                and node.args[0].id == parameter)
    for node in ast.walk(tree):
        if (isinstance(node, ast.BinOp) and isinstance(node.op, ast.Div) and parameter_path(node.left)
                and isinstance(node.right, ast.Constant) and node.right.value == "eval_config.yaml"):
            return True
        if (isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute) and node.func.attr == "joinpath"
                and parameter_path(node.func.value) and len(node.args) == 1 and not node.keywords
                and isinstance(node.args[0], ast.Constant) and node.args[0].value == "eval_config.yaml"):
            return True
    return False


def safe_loaded_loader(candidate):
    """Never import an arbitrary discovered module or infer an unseen call shape."""
    module = sys.modules.get(candidate["module"])
    function = module.__dict__.get(LOADER_NAME) if module is not None else None
    if (not inspect.isfunction(function) or function.__module__ != candidate["module"]
            or hasattr(function, "__wrapped__") or candidate["decorated"]
            or candidate["kind"] != "FunctionDef"):
        return None, "loader_not_already_loaded_as_an_undecorated_plain_function"
    file = inspect.getsourcefile(function)
    if file is None or Path(file).resolve() != Path(candidate["actual_path"]).resolve() or digest(Path(file).read_bytes()) != candidate["sha256"]:
        return None, "live_loader_source_identity_does_not_match_scanned_package"
    signature = inspect.signature(function, eval_str=False)
    parameters = list(signature.parameters.values())
    if len(parameters) != 1 or parameters[0].kind not in {
            inspect.Parameter.POSITIONAL_ONLY, inspect.Parameter.POSITIONAL_OR_KEYWORD, inspect.Parameter.KEYWORD_ONLY}:
        return None, "observed_loader_signature_is_not_one_unambiguous_path_argument"
    node = ast.parse(inspect.getsource(function)).body[0]
    if not config_path_role(node, parameters[0].name):
        return None, "observed_source_does_not_establish_argument_as_root_eval_config_path"
    builtin_calls = {"dict", "isinstance", "int", "float", "str", "bool", "len", "set", "sorted", "print",
                     "ValueError", "TypeError", "RuntimeError"}
    permitted_methods = {"exists", "is_file", "read_text", "read_bytes", "get", "items", "keys", "values", "joinpath"}
    for instruction in ast.walk(node):
        if isinstance(instruction, (ast.Import, ast.ImportFrom, ast.Global, ast.Nonlocal, ast.Await, ast.Yield, ast.YieldFrom)):
            return None, "observed_loader_body_contains_unreviewed_import_or_dynamic_scope"
        if isinstance(instruction, ast.Constant) and isinstance(instruction.value, str):
            lowered = instruction.value.lower()
            if any(term in lowered for term in ("tasks.jsonl", "snapshots/", "agent.yaml", "test_patch", "reference_patch")):
                return None, "observed_loader_body_mentions_data_outside_root_eval_config"
        if not isinstance(instruction, ast.Call):
            continue
        call = instruction.func
        if isinstance(call, ast.Name):
            if call.id == "Path" and function.__globals__.get("Path") is Path:
                continue
            if call.id not in builtin_calls or function.__globals__.get(call.id, getattr(builtins, call.id)) is not getattr(builtins, call.id):
                return None, "observed_loader_calls_unreviewed_helper_or_constructor"
        elif isinstance(call, ast.Attribute):
            if isinstance(call.value, ast.Name) and call.value.id == "yaml" and call.attr == "safe_load":
                yaml_module = function.__globals__.get("yaml")
                if yaml_module is None or getattr(yaml_module, "__name__", None) != "yaml":
                    return None, "loader_yaml_binding_is_not_the_observed_standard_module"
                continue
            if isinstance(call.value, ast.Name) and call.value.id in {"logger", "logging"} and call.attr in {"debug", "info", "warning", "error"}:
                continue
            if call.attr not in permitted_methods:
                return None, "observed_loader_calls_unreviewed_method"
        else:
            return None, "observed_loader_has_a_dynamic_call_target"
    return function, None


def json_return(value):
    """Retain actual values/types; never coerce loader values into a target schema."""
    if value is None or type(value) in {str, bool, int}:
        return value
    if type(value) is float and math.isfinite(value):
        return value
    if type(value) is list:
        return [json_return(item) for item in value]
    if type(value) is dict and all(type(key) is str for key in value):
        return {key: json_return(item) for key, item in value.items()}
    raise ValueError("Actual loader return is not a safe plain JSON value")


def probe_loader(candidate, destination):
    function, reason = safe_loaded_loader(candidate)
    if function is None:
        return {"status": "source_only_no_guessed_call", "reason": reason, "candidate": candidate,
                "loader_called": False, "observations": []}
    signature = inspect.signature(function, eval_str=False)
    parameter = next(iter(signature.parameters.values()))
    results = []
    for condition in ("absent_root_eval_config", "positive_root_eval_config"):
        with tempfile.TemporaryDirectory(prefix="neural_swe_budget_probe_") as temporary:
            directory = Path(temporary)
            if condition == "positive_root_eval_config":
                (directory / "eval_config.yaml").write_bytes(POSITIVE_CONFIG_BYTES)
            arguments = (directory,) if parameter.kind == inspect.Parameter.POSITIONAL_ONLY else ()
            keywords = {} if arguments else {parameter.name: directory}
            signature.bind(*arguments, **keywords)
            captured = io.StringIO()
            row = {"condition": condition, "source_identity": candidate, "observed_signature": str(signature),
                   "input_config_sha256": digest(POSITIVE_CONFIG_BYTES) if condition == "positive_root_eval_config" else None,
                   "input_config_values": POSITIVE_CONFIG if condition == "positive_root_eval_config" else None}
            try:
                with redirect_stdout(captured), redirect_stderr(captured):
                    value = function(*arguments, **keywords)
                row.update(status="returned", actual_return=json_return(value), return_type=type(value).__name__,
                           mapping_value_types={key: type(item).__name__ for key, item in value.items()} if type(value) is dict else None)
            except Exception as error:
                row.update(status="probe_exception", error_type=type(error).__name__, error=str(error))
            row["captured_diagnostics_sha256"] = digest(captured.getvalue().encode())
            row["temporary_directory_members_after_call"] = sorted(path.relative_to(directory).as_posix() for path in directory.rglob("*"))
            results.append(row)
    report = {"status": "actual_loaded_config_only_callable_probed", "candidate": candidate,
              "loader_called": True, "observations": results,
              "complete_schema_zero_negative_unknown_key_or_environment_precedence_verified": False}
    Path(destination).write_text(json.dumps(report, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    return report


def run_eval_config_source_probe(run_root, plan, checkpoint_exporter):
    run_root = Path(run_root)
    destination = run_root / "eval_config_source_probe.json"
    if destination.exists():
        raise FileExistsError("Preserve the previous library probe; use a new run")
    report = {"created_utc": datetime.now(timezone.utc).isoformat(), "status": "started",
        "scope": "installed_organizer_eval_config_source_and_optional_config_only_CPU_probe",
        "expected_packages": plan["expected_core_packages"], "model_inference_executed": False,
        "evaluator_or_grader_run_executed": False, "task_snapshot_or_reference_files_accessed": False,
        "arbitrary_discovered_modules_imported": False, "main_submission_selected": False}
    def save():
        destination.write_text(json.dumps(report, indent=2, allow_nan=False) + "\n", encoding="utf-8")
        checkpoint_exporter(run_root)
    save()
    try:
        receipt = json.loads((run_root / "environment_receipt.json").read_bytes())
        if receipt.get("mode") != "cpu_controls" or receipt.get("status") != "ready_for_cpu_controls" or receipt.get("cuda_device_count") != 0:
            raise ValueError("The package source probe requires the retained ready CPU environment")
        sources = scan_organizer_sources(run_root / "source_inventory", plan["expected_core_packages"])
        report.update(status="installed_package_source_collected", source_file_count=sources["file_count"],
                      source_bytes=sources["source_bytes"], definitions=sources["definitions"],
                      inventory_sha256=digest((run_root / "source_inventory/package_python_inventory.json").read_bytes()))
        save()
        candidates = [row for row in sources["definitions"] if row["name"] == LOADER_NAME]
        if len(candidates) != 1:
            report.update(status="source_only_loader_not_found_or_ambiguous", loader_definition_count=len(candidates),
                          loader_called=False, loader_probes=[])
        else:
            actual = probe_loader(candidates[0], run_root / "loader_parse_probes.json")
            report.update(status="completed_library_contract_probe", loader_called=actual["loader_called"], loader_probes=[actual])
            if not actual["loader_called"]:
                (run_root / "loader_parse_probes.json").write_text(json.dumps(actual, indent=2) + "\n", encoding="utf-8")
        save()
        return report
    except Exception as error:
        report.update(status="source_probe_exception", error_type=type(error).__name__, error=str(error))
        (run_root / "source_probe_exception.txt").write_text(traceback.format_exc(), encoding="utf-8")
        save()
        raise


In [ ]:
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    result = run_eval_config_source_probe(RUN, PLAN, export_run)
    print('EVAL_CONFIG_SOURCE_PROBE=' + json.dumps(result))
    export_run(RUN)
else:
    print('No installed-source probe or loader invocation executed locally')
